# DATETIMEEVENTS - Silver Layer

This notebook cleans and validates the MIMIC-IV `datetimeevents` Bronze table before storing it in the Silver layer.

`DATETIMEEVENTS` contains ICU events where the recorded value is itself a date or time. Each record can be linked to `D_ITEMS` through `itemid` and to `CAREGIVER` through `caregiver_id`.

The Silver layer focuses on missing values, duplicate records and key relationships while preserving the original datetime event structure.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

datetimeevents = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.datetimeevents_raw"
)

print(f"Rows: {datetimeevents.count():,}")
print(f"Columns: {len(datetimeevents.columns)}")

check_columns = [
    "subject_id",
    "hadm_id",
    "stay_id",
    "caregiver_id",
    "charttime",
    "storetime",
    "itemid",
    "value",
    "valueuom",
    "warning"
]

total = datetimeevents.count()

for column in check_columns:
    nulls = datetimeevents.filter(F.col(column).isNull()).count()

    if nulls > 0:
        pct = (nulls / total) * 100
        print(f"{column}: {nulls:,} nulls ({pct:.2f}%)")

### Missing Value Findings

The main patient, admission, ICU stay, caregiver, item and datetime fields are all complete.

No missing values were found in the fields checked, so the next step is to confirm there are no duplicate event records and that the item and caregiver relationships are valid.

In [0]:
# Check duplicates and lookup relationships
duplicate_rows = (
    datetimeevents
    .groupBy(
        "subject_id",
        "hadm_id",
        "stay_id",
        "caregiver_id",
        "charttime",
        "itemid",
        "value"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

d_items = spark.table("workspace.silver.d_items")
caregiver = spark.table("workspace.silver.caregiver")

unmatched_items = (
    datetimeevents
    .select("itemid")
    .distinct()
    .join(d_items.select("itemid"), on="itemid", how="left_anti")
    .count()
)

unmatched_caregivers = (
    datetimeevents
    .select("caregiver_id")
    .distinct()
    .join(caregiver.select("caregiver_id"), on="caregiver_id", how="left_anti")
    .count()
)

print(f"Potential duplicate event groups: {duplicate_rows:,}")
print(f"Unmatched itemid values: {unmatched_items:,}")
print(f"Unmatched caregiver_id values: {unmatched_caregivers:,}")

### Data Quality Findings

- No duplicate event groups were found.
- All `itemid` values successfully match `D_ITEMS`.
- All `caregiver_id` values successfully match `CAREGIVER`.
- The main event fields are complete, so only very light Silver cleaning is needed.

In [0]:
# Light text cleaning
datetimeevents_silver = (
    datetimeevents
    .withColumn(
        "valueuom",
        F.when(F.trim(F.col("valueuom")) == "", None)
         .otherwise(F.trim(F.col("valueuom")))
    )
)

print(f"DATETIMEEVENTS Silver: {datetimeevents_silver.count():,} rows")
display(datetimeevents_silver.limit(10))

### Silver Transformation Check

`DATETIMEEVENTS` is preserving the recorded datetime values and their associated unit labels correctly.

The table has complete key fields, no duplicate event groups and valid links to both `D_ITEMS` and `CAREGIVER`, so no further transformation is needed.

In [0]:
# Save Silver table
(
    datetimeevents_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.datetimeevents")
)

bronze_count = spark.table("workspace.bronze.datetimeevents_raw").count()
silver_count = spark.table("workspace.silver.datetimeevents").count()

print(f"Bronze: {bronze_count:,}")
print(f"Silver: {silver_count:,}")
print(f"Match: {bronze_count == silver_count}")

### Silver Result

`DATETIMEEVENTS` was successfully written to the Silver layer with all Bronze records preserved.

The table required only light text standardisation, with complete key fields, no duplicate event groups and valid relationships to the supporting lookup tables.